## Import libraries

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import time
from plotly.subplots import make_subplots
import os
from sklearn.metrics import confusion_matrix, accuracy_score
import kaleido

In [ ]:
rep_path='../../../'

### Define the function to draw taxels

In [ ]:
def drawBoxesAndBoundary():
    # Draw the 16 square boxes
    corners = [-0.75, -5.25, -9.75, -14.25]
    shapes = []
    for i in corners:
        for j in corners:
            shapes.append(
                dict(
                    type="rect",
                    x0=i-3,
                    y0=j-3,
                    x1=i,
                    y1=j,
                    line=dict(color="black", width=1)
                )
            )

    # Define the boundary
    xvec = np.linspace(-18, 0, 25)
    yvec = np.linspace(-18, 0, 25)
    boundary = go.Scatter(
        x=[min(xvec), min(xvec), max(xvec), max(xvec), min(xvec)],
        y=[min(yvec), max(yvec), max(yvec), min(yvec), min(yvec)],
        mode="lines",
        line=dict(color="black", width=1)
    )

    return shapes, boundary

# Call the function to get shapes and boundary
shapes, boundary = drawBoxesAndBoundary()

# Create a new figure
fig = go.Figure()

# Add shapes (boxes)
for shape in shapes:
    fig.add_shape(shape)

# Add boundary
fig.add_trace(boundary)

# Update layout
fig.update_layout(
    xaxis=dict(
        scaleanchor="y",
        scaleratio=1,
        range=[-18, 0]
    ),
    yaxis=dict(
        range=[-18, 0]
    ),
    showlegend=False
)

# Save the plot as SVG
# fig.write_image(os.path.join(rep_path,'Code/Machine learning modelling/exported_figures/Boxes.svg'))

# Add a delay
time.sleep(1)

# Display the plot
fig.show()


# Test error results plots

In [ ]:
test_error_dataset = pd.read_excel(os.path.join(rep_path,'Data/Results/Single_Point_Contact_Estimation_Results/test_error_results.xlsx'), header=0)
test_error_dataset[[8, 9, 10]] = (test_error_dataset.iloc[:, :3].sub(test_error_dataset.iloc[:, 4:7].values)).abs()
test_error_dataset_force_filtered = test_error_dataset[test_error_dataset.iloc[:, 3] == 1]  
test_error_dataset_force_filtered.iloc[:, 2] *= -1
test_error_dataset_force_filtered = test_error_dataset_force_filtered[test_error_dataset_force_filtered.iloc[:, 2] <= 2]  

In [ ]:
# Sort the DataFrame by the 17th and 18th columns in descending order
test_error_dataset_force_filtered_sorted = test_error_dataset_force_filtered.sort_values(
    by=[test_error_dataset_force_filtered.columns[0], test_error_dataset_force_filtered.columns[1]],
    ascending=[False, False]
).reset_index(drop=True)

# Initialize an index array to mark rows for deletion
rows_to_delete = np.zeros(len(test_error_dataset_force_filtered_sorted), dtype=bool)

# Loop through the DataFrame to check for consecutive equal values in the 17th and 18th columns
for i in range(len(test_error_dataset_force_filtered_sorted) - 2):
    # Check for 3 consecutive equal values in column 17
    if not (
        test_error_dataset_force_filtered_sorted.iloc[i, 0] == test_error_dataset_force_filtered_sorted.iloc[i + 1, 0] == test_error_dataset_force_filtered_sorted.iloc[i + 2, 0]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

    # Check for 3 consecutive equal values in column 18
    if not (
        test_error_dataset_force_filtered_sorted.iloc[i, 1] == test_error_dataset_force_filtered_sorted.iloc[i + 1, 1] == test_error_dataset_force_filtered_sorted.iloc[i + 2, 1]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

# Delete the marked rows
test_error_dataset_force_filtered_cleaned = test_error_dataset_force_filtered_sorted[~rows_to_delete]


In [ ]:
# Optionally, display the cleaned DataFrame
print("Size of test_error_dataset_force_filtered:", test_error_dataset_force_filtered.shape)
print("Size of test_error_dataset_force_filtered:", test_error_dataset_force_filtered_cleaned.shape)

Touch binary classification accuracy result

In [ ]:
# This code filters out transient sensor noise by deleting any force readings that do not appear in a stable, consecutive streak of at least three identical values.
test_error_dataset_sorted = test_error_dataset.sort_values(
    by=[test_error_dataset.columns[0], test_error_dataset.columns[1]],
    ascending=[False, False]
).reset_index(drop=True)

# Initialize an index array to mark rows for deletion
rows_to_delete = np.zeros(len(test_error_dataset_sorted), dtype=bool)

# Loop through the DataFrame to check for consecutive equal values
for i in range(len(test_error_dataset_sorted) - 2):
    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_sorted.iloc[i, 0] == test_error_dataset_sorted.iloc[i + 1, 0] == test_error_dataset_sorted.iloc[i + 2, 0]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_sorted.iloc[i, 1] == test_error_dataset_sorted.iloc[i + 1, 1] == test_error_dataset_sorted.iloc[i + 2, 1]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

# Delete the marked rows
test_error_dataset_sorted_cleaned = test_error_dataset_sorted[~rows_to_delete]

# Extract the target and predicted classification output
y_true = test_error_dataset_sorted_cleaned.iloc[:, 3].values  # True target values 
y_pred = test_error_dataset_sorted_cleaned.iloc[:, 7].values  # Predicted output values

# Threshold the predicted values at 0.5
y_pred_binary = (y_pred >= 0.5).astype(int)

# Generate the confusion matrix using sklearn
cm = confusion_matrix(y_true, y_pred_binary)
print("Confusion Matrix:")
print(cm)

# Calculate classification accuracy
accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Classification Accuracy: {accuracy:.4f}")

In [ ]:
# Calculate mean of x, y, force errors
mean_x_error = test_error_dataset_force_filtered_cleaned.iloc[:, 8].mean()
mean_y_error = test_error_dataset_force_filtered_cleaned.iloc[:, 9].mean()
mean_force_error = test_error_dataset_force_filtered_cleaned.iloc[:, 10].mean()

print("Mean of X error:", mean_x_error)
print("Mean of Y error:", mean_y_error)
print("Mean of Force error:", mean_force_error)

In [ ]:
# Define the ranges
ranges = [0.02, 0.09, 0.5, 1, 1.5, 2]

# Create labels for the ranges
labels = ['[0.02, 0.09)', '[0.09, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2)']

In [ ]:
# --- PRE-CALCULATE INDIVIDUAL EUCLIDEAN DISTANCES ---
# Using your existing abs error columns (8 and 9)
test_error_dataset_force_filtered_cleaned['euclidean_dist'] = np.sqrt(
    test_error_dataset_force_filtered_cleaned.iloc[:, 8]**2 + 
    test_error_dataset_force_filtered_cleaned.iloc[:, 9]**2
)

# Results for the test dataset

In [ ]:
# Clean up by removing the data from inside the edges:

# Define conditions for filtering
sor = (
    (test_error_dataset_force_filtered_cleaned.iloc[:, 0] < -17.25) | 
    (test_error_dataset_force_filtered_cleaned.iloc[:, 0] > -0.75) | 
    (test_error_dataset_force_filtered_cleaned.iloc[:, 1] < -17.25) | 
    (test_error_dataset_force_filtered_cleaned.iloc[:, 1] > -0.75)
)

filtered_indices_edge_sensor = sor

filtered_indices_inbetween_taxels = (
    (~filtered_indices_edge_sensor) & 
    (
        (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] < -3.75) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] > -5.25)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] < -8.25) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] > -9.75)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] < -12.75) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] > -14.25)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] < -3.75) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] > -5.25)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] < -8.25) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] > -9.75)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] < -12.75) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] > -14.25)
        )
    )
)

filtered_indices_inside_taxels = (
    (~filtered_indices_edge_sensor) & 
    (~filtered_indices_inbetween_taxels) & 
    (
        (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] <= -1.5) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] >= -3)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] <= -6) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] >= -7.5)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] <= -10.5) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] >= -12)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] <= -15) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 0] >= -16.5)
        )
    ) & (
        (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] <= -1.5) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] >= -3)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] <= -6) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] >= -7.5)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] <= -10.5) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] >= -12)
        ) | (
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] <= -15) & 
            (test_error_dataset_force_filtered_cleaned.iloc[:, 1] >= -16.5)
        )
    )
)

filtered_indices_edge_taxels = (
    (~filtered_indices_edge_sensor) & 
    (~filtered_indices_inbetween_taxels) & 
    (~filtered_indices_inside_taxels)
)

filtered_indices_inside_edge_sensor = (
    (~filtered_indices_edge_sensor)
)

filtered_indices_sensor_plus_outside = np.ones(len(test_error_dataset_force_filtered_cleaned), dtype=bool)

In [ ]:
# --- CALCULATE METRICS PER REGION ---

def get_region_metrics(df, indices, name):
    subset = df.loc[indices]
    if subset.empty:
        return None
    
    abs_x = subset.iloc[:, 8]
    abs_y = subset.iloc[:, 9]
    abs_f = subset.iloc[:, 10]
    dist  = subset['euclidean_dist']
    
    return {
        "name": name,
        "MEE": dist.mean(),
        "MAE_X": abs_x.mean(),
        "MAE_Y": abs_y.mean(),
        "MAE_Force": abs_f.mean(),
        "Spatial_Bias": np.sqrt(abs_x.mean()**2 + abs_y.mean()**2)
    }

In [ ]:
# Compute for each group
regions_data = [
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inside_taxels, "Inside Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inbetween_taxels, "In-between Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_edge_taxels, "Edge Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inside_edge_sensor, "Sensor"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_sensor_plus_outside, "Sensor plus outside edge")
]

# Print Results
for res in regions_data:
    if res:
        print(f"--- {res['name']} ---")
        print(f"MEE: {res['MEE']:.4f} mm | L2 Norm of MAE: {res['Spatial_Bias']:.4f} mm")
        print(f"MAE in X-direction: {res['MAE_X']:.4f} mm | MAE in Y-direction: {res['MAE_Y']:.4f} mm")
        print(f"MAE Force: {res['MAE_Force']:.4f} N\n")

In [ ]:
# --- UPDATE PLOT DATA ---
regions = [r['name'] for r in regions_data]
mean_mee_errors = [r['MEE'] for r in regions_data]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=regions,
    y=mean_mee_errors,
    name='Mean Euclidean Error',
    text=[f'{v:.2f}' for v in mean_mee_errors],
    textposition='auto',
    width=0.4,
    marker=dict(color='purple', opacity=0.7)
))

fig.update_layout(
    title='Mean Euclidean Error (MEE) by Region',
    xaxis_title='Regions',
    yaxis_title='MEE [mm]',
    template='plotly_white'
)
fig.show()

## Force prediction error for different regions

In [ ]:
# --- 1. Prepare Data for Force Plotting ---
# Extracting names and MAE Force values from our regions_data list
# This includes the "Sensor plus outside edge" category you requested
plot_regions = [res['name'] for res in regions_data if res is not None]
mae_force_values = [res['MAE_Force'] for res in regions_data if res is not None]

# --- 2. Create the Force Bar Chart ---
fig_force = go.Figure()

fig_force.add_trace(go.Bar(
    x=plot_regions,
    y=mae_force_values,
    name='Force MAE',
    text=[f'{val:.2f}' for val in mae_force_values],
    textposition='auto',
    width=0.4,
    marker=dict(color='green', opacity=0.6) 
))

# Update layout
fig_force.update_layout(
    title='Force Prediction Error (MAE) for Different Regions',
    xaxis_title='Regions',
    yaxis_title='Mean Absolute Error [N]',
    template='plotly_white'
)

# Display the plot
fig_force.show()

# --- 3. Print Results (Matching your requested format) ---
# We pull these directly from regions_data to ensure mathematical consistency
print("FORCE ERROR ANALYSIS\n" + "="*30)

# Helper to find specific region by name in our list
def print_stat(name):
    match = next((item for item in regions_data if item["name"] == name), None)
    if match:
        print(f"Total Force Error for {name}: {match['MAE_Force']:.4f} N")

print_stat("Edge Taxels") # You previously called this 'Edge Sensor Region' in print
print_stat("In-between Taxels")
print_stat("Edge Taxels")
print_stat("Inside Taxels")
print_stat("Sensor")
print_stat("Sensor plus outside edge")

### Plot violin plot for test dataset (location)

In [ ]:
data_set_locations="sensor_area"

In [ ]:
# Define the ranges
ranges = [0.02, 0.5, 1, 1.5, 2]
# Create labels for the ranges
labels = ['[0.02, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2]']


if data_set_locations=="sensor_plus_edges":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[:] 
elif data_set_locations=="sensor_area":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[filtered_indices_inside_edge_sensor]

# Create a new column 'category' based on the ranges
filtered_test_dataset['category'] = pd.cut(filtered_test_dataset.iloc[:, 2], bins=ranges, labels=labels, right=False)

# Specify x values
x = filtered_test_dataset['category']  # Categorical variable created based on the ranges

y_9th_column = filtered_test_dataset.iloc[:, 4]-filtered_test_dataset.iloc[:, 0]   
y_10th_column = filtered_test_dataset.iloc[:, 5]-filtered_test_dataset.iloc[:, 1]
x_error = y_9th_column
y_error = y_10th_column

# Calculate the square root of the sum of squares of the two columns
y_sqrt = np.sqrt(x_error ** 2 + y_error ** 2)

# Filter out negative values for y_sqrt
y_sqrt = y_sqrt[y_sqrt >= 0]

fig = go.Figure()

fig.add_trace(go.Violin(x=x, y=y_9th_column, points=False, box_visible = True, meanline_visible = False, side='negative', x0= -3, line_color='blue', opacity=0.6, name='X Error', width=0.9))
fig.add_trace(go.Violin(x=x, y=y_10th_column, points=False, box_visible = True, meanline_visible = False, side='negative', x0 = -2, line_color='red', opacity=0.6, name='Y Error', width=0.9))
fig.add_trace(go.Violin(x=x, y=y_sqrt, points=False, box_visible = True, meanline_visible = False, side='positive', line_color='purple', opacity=0.6, name='Total Error', width=0.9, spanmode='hard'))

fig.update_layout(
    title='Location Prediction Error',
    xaxis=dict(title='Force Range [N]', categoryorder='array', categoryarray=labels),
    # yaxis=dict(title='Location Prediction Error'),
    showlegend=True  # Show legend to distinguish between columns
)

fig.update_layout(width=600,height=400,
    yaxis=dict(
        title='Error [mm]',
        range=[-2.5, 2.5]
    )
)

# Display the plot
fig.show()

In [ ]:
# Save the plot as SVG
fig.write_image('exported_figures/Violin_test_location_error_results.svg')

### Violin force

In [ ]:
# Define the ranges
ranges = [0.02, 0.5, 1, 1.5, 2]

# Create labels for the ranges
labels = ['[0.02, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2]']

# Cut the data into four parts based on the ranges of column 3
filtered_test_dataset ['category'] = pd.cut(filtered_test_dataset .iloc[:, 2], bins=ranges, labels=labels, right=False)

# Specify x values
x = filtered_test_dataset ['category']  # Categorical variable created based on the ranges

# Specify y values
F_abs_error=filtered_test_dataset .iloc[:, 10]
F_signed_error=-filtered_test_dataset.iloc[:, 6]-filtered_test_dataset .iloc[:, 2]

# Create violin plots 
fig = go.Figure()
#0A7FF4

fig.add_trace(go.Violin(x=x, y=F_abs_error, points=False, box_visible = False, meanline_visible = True, side='positive', line_color='#4E99EC', opacity=0.85, name='Absolute Force Error', width=1, spanmode='hard'))
fig.add_trace(go.Violin(x=x, y=F_signed_error, points=False, box_visible = False, meanline_visible = True, side='negative', line_color='#3A2D76', opacity=0.85, name='Force Error', width=1, spanmode='hard'))


fig.update_layout(
    title='Force Prediction Error',
    xaxis=dict(title='Force Range [N]', categoryorder='array', categoryarray=labels),
    # yaxis=dict(title='Location Prediction Error'),
    showlegend=True,  # Show legend to distinguish between columns
    plot_bgcolor='white'
)

fig.update_layout(width=800,height=400,
    yaxis=dict(
        title='Error [N]',
        range=[-1,1]
    )
)

# Save the plot as SVG
# fig.write_image('exported_figures/Violin_test_force_error_results_Single_Contact.svg')

# Add a delay
time.sleep(1)

# Display the plot
fig.show()

## Plot Heatmap and Scatter plot for test dataset

In [ ]:
filtered_test_dataset = filtered_test_dataset.drop(columns=['category'])
# Group the dataset by physical coordinates (Actual X and Actual Y) and calculate the average error for each specific location to visualize spatial performance across the sensor surface.
error_data_grouped_filtered = filtered_test_dataset.groupby([filtered_test_dataset.columns[0], filtered_test_dataset.columns[1]]).mean().reset_index()

error_data_grouped_filtered.iloc[:, 0] = error_data_grouped_filtered.iloc[:, 0].round(2)
error_data_grouped_filtered.iloc[:, 1] = error_data_grouped_filtered.iloc[:, 1].round(2)

# Compute error values as the square root of the sum of squares of the X and Y column values
error_data_grouped_filtered['z'] = np.sqrt(error_data_grouped_filtered.iloc[:, 8]*error_data_grouped_filtered.iloc[:, 8] + error_data_grouped_filtered.iloc[:, 9]*error_data_grouped_filtered.iloc[:, 9])

# Apply the np.log2 function to transform the error values to a logarithmic scale
error_data_grouped_filtered['z'] = np.log2(error_data_grouped_filtered['z'])

### Scatter plot location

In [ ]:
# 1. Calculate the Euclidean distance for every single row in the RAW dataset
# Column 8 is |dx|, Column 9 is |dy|
filtered_test_dataset['individual_dist'] = np.sqrt(
    filtered_test_dataset.iloc[:, 8]**2 + 
    filtered_test_dataset.iloc[:, 9]**2
)

# 2. NOW group by location and take the mean of those distances
error_data_grouped = filtered_test_dataset.groupby([
    filtered_test_dataset.columns[0], 
    filtered_test_dataset.columns[1]
]).mean().reset_index()

# 3. The 'z' value for your plot is now the average of the distances
error_data_grouped['z'] = error_data_grouped['individual_dist']

# 4. Proceed with log transformation for visualization
error_data_grouped_filtered_log = error_data_grouped.copy()
error_data_grouped_filtered_log['z'] = np.log2(error_data_grouped['z'] + 1e-6) # add small epsilon to avoid log(0)

In [ ]:
import plotly.express as px
import numpy as np

fig = px.scatter(error_data_grouped_filtered_log['z'],x=error_data_grouped_filtered_log.iloc[:, 0],  # X coordinates
    y=error_data_grouped_filtered_log.iloc[:, 1],color=error_data_grouped_filtered_log['z'],  # Y coordinate)
    hover_name=error_data_grouped_filtered_log['z']
)

fig.update_layout(coloraxis_colorbar=dict(
            title='Absolute Error [mm]',  # Add color bar
            tickvals=[i for i in range(-1, 5)],  # Set tick values as powers of 2
            ticktext=[f'{2**i}' for i in range(-1, 5)]  # Set tick labels as '2^i'
        ),
)

# Update layout for the scatter plot
fig.update_layout(width=600,height=400,
    title='Scatter Plot of Location Prediction Error (MEE)',
    xaxis_title='X Location [mm]',
    yaxis_title='Y Location [mm]',
    xaxis=dict(
        scaleanchor="y",
        scaleratio=1,
        range=[-18, 0]
    ),
    yaxis=dict(
        range=[-18, 0]
    ),
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white',  # Set paper background color to white
    showlegend=False
)

fig.update_layout(coloraxis_colorbar=dict(
    title=dict(text="Location error [mm]"),
))
fig.update_traces(marker=dict(size=10))

def drawBoxesAndBoundary():
    # Draw the 16 square boxes
    corners = [-0.75, -5.25, -9.75, -14.25]
    shapes = []
    for i in corners:
        for j in corners:
            shapes.append(
                dict(
                    type="rect",
                    x0=i-3,
                    y0=j-3,
                    x1=i,
                    y1=j,
                    line=dict(color="black", width=1)
                )
            )

    # Define the boundary
    xvec = np.linspace(-18, 0, 25)
    yvec = np.linspace(-18, 0, 25)
    boundary = go.Scatter(
        x=[min(xvec), min(xvec), max(xvec), max(xvec), min(xvec)],
        y=[min(yvec), max(yvec), max(yvec), min(yvec), min(yvec)],
        mode="lines",
        line=dict(color="black", width=1)
    )

    return shapes, boundary


# Call the function to get shapes and boundary
shapes, boundary = drawBoxesAndBoundary()
# Add shapes (boxes) to the figure
for shape in shapes:
    fig.add_shape(shape)

# Add boundary to the figure
fig.add_trace(boundary)

# Add a delay
time.sleep(2)


fig.show()
# Save the plot as SVG
# fig.write_image('exported_figures/Scatter_test_location_error_results.svg')

### Heatmap Force

In [ ]:
# Compute error values as the normal force error value
error_data_grouped_filtered['z'] = error_data_grouped_filtered.iloc[:, 10].abs()
error_data_grouped_filtered['z'] = np.log2(error_data_grouped_filtered['z'])

### Scatter plot force

In [ ]:
# Create a new figure for the scatter plot
fig = go.Figure()
original_colorscale = px.colors.sequential.Turbo
truncated_colorscale = original_colorscale[:int(len(original_colorscale) * 0.7)]


# Add scatter plot
fig.add_trace(go.Scatter(
    x=error_data_grouped_filtered.iloc[:, 0],  # X coordinates
    y=error_data_grouped_filtered.iloc[:, 1],  # Y coordinates
    mode='markers',  # Scatter plot with markers only
    marker=dict(
        size=10,  # Size of markers
        color=error_data_grouped_filtered['z'],  # Color of markers based on 'z' values
        colorscale=truncated_colorscale,  # Use the custom color scale
        showscale=True,  # Show color scale
        colorbar=dict(
            title='Absolute Error [N]',  # Add color bar
            tickvals=[i for i in range(-4, 1)],  # Set tick values as powers of 2
            ticktext=[f'{2**i}' for i in range(-4, 1)]  # Set tick labels as '2^i'
        ),
    )
    
))

# Update layout for the scatter plot
fig.update_layout(width=600,height=400,
    title='Scatter Plot of Force Prediction Error',
    xaxis_title='X Location [mm]',
    yaxis_title='Y Location [mm]',
    xaxis=dict(
        scaleanchor="y",
        scaleratio=1,
        range=[-18, 0]
    ),
    yaxis=dict(
        range=[-18, 0]
    ),
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white',  # Set paper background color to white
    showlegend=False
)

# Call the function to get shapes and boundary
shapes, boundary = drawBoxesAndBoundary()

# Add shapes (boxes) to the figure
for shape in shapes:
    fig.add_shape(shape)

# Add boundary to the figure
fig.add_trace(boundary)# Call the function to get shapes and boundary
shapes, boundary = drawBoxesAndBoundary()

# Add shapes (boxes) to the figure
for shape in shapes:
    fig.add_shape(shape)

# Add boundary to the figure
fig.add_trace(boundary)

# Save the plot as SVG
# fig.write_image('exported_figures/Scatter_test_force_error_results.svg')

# Add a delay
time.sleep(1)

# Display the plot
fig.show()